# 🚗 Tesla Stock Price Prediction
## Deep Learning: SimpleRNN vs LSTM

**Dataset:** TSLA.csv — 2416 rows | 2010-06-29 to 2019-12-31  
**Target:** Predict Adj Close price for 1d, 5d, 10d horizons  
**Models:** SimpleRNN and LSTM  
**Metrics:** MSE, RMSE, MAE, MAPE

---
## 0️⃣ Install & Import Libraries

In [ ]:
# Uncomment below if running for the first time
# !pip install tensorflow-cpu scikit-learn scikeras pandas matplotlib seaborn plotly streamlit joblib -q

import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import os, joblib

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.model_selection import GridSearchCV

import tensorflow as tf
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import SimpleRNN, LSTM, Dense, Dropout, BatchNormalization, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from scikeras.wrappers import KerasRegressor

print('TensorFlow version:', tf.__version__)
os.makedirs('../models',  exist_ok=True)
os.makedirs('../reports', exist_ok=True)

%matplotlib inline
plt.rcParams['figure.dpi'] = 110
sns.set_style('whitegrid')
print('✅ All libraries loaded!')

---
## 1️⃣ Load & Explore Dataset

In [ ]:
# ── Load ──────────────────────────────────────────────────
df_raw = pd.read_csv('../data/TSLA.csv', parse_dates=['Date'], index_col='Date')
df_raw.sort_index(inplace=True)

print('Shape     :', df_raw.shape)
print('Date range:', df_raw.index.min().date(), '→', df_raw.index.max().date())
df_raw.head(10)

In [ ]:
print('\n── Data Types ─────────────────────────')
print(df_raw.dtypes)
print('\n── Descriptive Statistics ─────────────')
df_raw.describe().round(2)

In [ ]:
print('── Missing Values ──────────────────────')
print(df_raw.isnull().sum())
print('Total missing:', df_raw.isnull().sum().sum())

---
## 2️⃣ Data Cleaning — Missing Value Handling

In [ ]:
"""
STRATEGY: Forward-Fill (ffill) + Back-Fill (bfill)

WHY NOT mean/median?
  → Stock data is time-ordered (sequential).
  → Mean imputation ignores the time structure and introduces
     unrealistic jumps in the series.

WHY forward-fill?
  → The most recent known price is the best estimate
     for a missing price (e.g., holidays, trading halts).
  → This preserves the temporal continuity of the series.
"""

df = df_raw.copy()
df = df.ffill().bfill()

assert df.isnull().sum().sum() == 0, '❌ Still missing!'
print('✅ No missing values. Dataset is clean.')
df.head()

---
## 3️⃣ Exploratory Data Analysis (EDA)

In [ ]:
# ── Plot 1: Closing Price ─────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(df.index, df['Adj Close'], color='steelblue', linewidth=1, label='Adj Close')
ax.set_title('Tesla Adjusted Closing Price (2010–2019)', fontsize=14)
ax.set_ylabel('Price (USD)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.legend(); plt.tight_layout()
plt.savefig('../reports/01_price.png', dpi=150)
plt.show()

In [ ]:
# ── Plot 2: Volume ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 2.5))
ax.bar(df.index, df['Volume'], color='slategray', alpha=0.7)
ax.set_title('Tesla Daily Trading Volume')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.tight_layout()
plt.savefig('../reports/02_volume.png', dpi=150)
plt.show()

In [ ]:
# ── Plot 3: OHLC Grid ──────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 7))
items  = [('Open','Open'), ('High','High'), ('Low','Low'), ('Adj Close','Adj Close')]
colors = ['forestgreen', 'tomato', 'slateblue', 'steelblue']
for ax, (col, title), color in zip(axes.ravel(), items, colors):
    ax.plot(df.index, df[col], color=color, linewidth=0.8)
    ax.set_title(title); ax.set_ylabel('USD')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.suptitle('Tesla OHLC Price History', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('../reports/03_ohlc.png', dpi=150)
plt.show()

In [ ]:
# ── Plot 4: Correlation Heatmap ───────────────────────────
fig, ax = plt.subplots(figsize=(7, 5))
cols = ['Open','High','Low','Close','Adj Close','Volume']
sns.heatmap(df[cols].corr(), annot=True, fmt='.2f', cmap='coolwarm', ax=ax, square=True)
ax.set_title('Feature Correlation Heatmap')
plt.tight_layout()
plt.savefig('../reports/04_correlation.png', dpi=150)
plt.show()
print('✅ Key insight: Open/High/Low/Close are all highly correlated → use Adj Close as target')

---
## 4️⃣ Feature Engineering

In [ ]:
# ── Add technical indicators ──────────────────────────────
df['MA_20']        = df['Adj Close'].rolling(20).mean()   # Short-term trend
df['MA_50']        = df['Adj Close'].rolling(50).mean()   # Long-term trend
df['Daily_Return'] = df['Adj Close'].pct_change()         # % daily change
df['Volatility']   = df['Daily_Return'].rolling(20).std() # Price stability

df.dropna(inplace=True)   # Remove rows with NaN from rolling windows
print(f'Shape after feature engineering: {df.shape}')
df.tail(5)

In [ ]:
# ── Plot: Price + Moving Averages ─────────────────────────
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(df.index, df['Adj Close'], color='steelblue', linewidth=1,   label='Adj Close')
ax.plot(df.index, df['MA_20'],     color='orange',    linewidth=1,   linestyle='--', label='MA 20')
ax.plot(df.index, df['MA_50'],     color='red',       linewidth=1,   linestyle='--', label='MA 50')
ax.set_title('Adj Close + Moving Averages'); ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.tight_layout()
plt.savefig('../reports/05_moving_avg.png', dpi=150)
plt.show()

In [ ]:
# ── Plot: Daily Return + Volatility ──────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(df['Daily_Return'], bins=80, kde=True, ax=axes[0], color='teal')
axes[0].set_title('Daily Return Distribution')
axes[1].plot(df.index, df['Volatility'], color='darkorange', linewidth=0.8)
axes[1].set_title('Rolling 20-Day Volatility')
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.tight_layout()
plt.savefig('../reports/06_volatility.png', dpi=150)
plt.show()

---
## 5️⃣ Data Preprocessing — Scaling & Sequences

In [ ]:
# ── Constants ─────────────────────────────────────────────
WINDOW     = 60    # 60 days look-back (≈ 3 months of trading)
TEST_RATIO = 0.20  # 80% train, 20% test
FEATURE    = 'Adj Close'

# ── Scale to [0, 1] ───────────────────────────────────────
"""
WHY MinMaxScaler?
  Neural networks work best with small input values.
  Stock prices range from $15 to $400+. Scaling to [0,1]
  makes gradients stable and training faster.
"""
scaler = MinMaxScaler(feature_range=(0, 1))
scaled = scaler.fit_transform(df[[FEATURE]].values)  # shape: (N, 1)

# Save scaler for all horizons
for h in [1, 5, 10]:
    joblib.dump(scaler, f'../models/scaler_h{h}.pkl')

print(f'Scaled shape: {scaled.shape}')
print(f'Min: {scaled.min():.4f}  Max: {scaled.max():.4f}')

In [ ]:
# ── Sequence Creator ──────────────────────────────────────
def create_sequences(scaled_data, window=60, horizon=1):
    """
    Sliding window approach.

    Example for window=3, horizon=1:
      Input:  [day1, day2, day3] → Output: day4
      Input:  [day2, day3, day4] → Output: day5
      ...

    Returns:
      X: shape (samples, window, 1)
      y: shape (samples,) for h=1  OR  (samples, horizon) for h>1
    """
    X, y = [], []
    for i in range(window, len(scaled_data) - horizon + 1):
        X.append(scaled_data[i - window:i, 0])
        y.append(scaled_data[i:i + horizon, 0])
    X = np.array(X)[..., np.newaxis]
    y = np.array(y)
    if horizon == 1:
        y = y.ravel()
    return X, y

def ts_split(X, y, test_ratio=0.20):
    """Chronological split — NO shuffle for time-series!"""
    split = int(len(X) * (1 - test_ratio))
    return X[:split], X[split:], y[:split], y[split:]

# Test with horizon=1
X1, y1       = create_sequences(scaled, WINDOW, 1)
Xtr1, Xte1, ytr1, yte1 = ts_split(X1, y1)
print(f'Horizon 1 → X_train: {Xtr1.shape}  X_test: {Xte1.shape}')
print(f'           y_train: {ytr1.shape}  y_test: {yte1.shape}')

---
## 6️⃣ Model Definitions

In [ ]:
def build_simple_rnn(units=64, dropout_rate=0.2, learning_rate=0.001, horizon=1):
    """
    SimpleRNN Architecture:
      Input(60,1) → RNN(units) → Dropout → RNN(units/2)
              → Dropout → Dense(32,relu) → Dense(horizon)

    LIMITATION: suffers from vanishing gradients on long sequences.
    """
    model = Sequential(name='SimpleRNN')
    model.add(Input(shape=(WINDOW, 1)))
    model.add(SimpleRNN(units, return_sequences=True))
    model.add(Dropout(dropout_rate))
    model.add(SimpleRNN(units // 2))
    model.add(Dropout(dropout_rate))
    model.add(Dense(32, activation='relu'))
    model.add(Dense(horizon))
    model.compile(optimizer=Adam(learning_rate), loss='mse', metrics=['mae'])
    return model


def build_lstm(units=64, dropout_rate=0.2, learning_rate=0.001, horizon=1):
    """
    LSTM Architecture:
      Input(60,1) → LSTM(units) → Dropout → LSTM(units/2)
               → Dropout → BatchNorm → Dense(32,relu) → Dense(horizon)

    ADVANTAGE: 3 gates (Forget/Input/Output) control what to remember.
    Solves vanishing gradient — retains long-term dependencies.
    """
    model = Sequential(name='LSTM')
    model.add(Input(shape=(WINDOW, 1)))
    model.add(LSTM(units, return_sequences=True))
    model.add(Dropout(dropout_rate))
    model.add(LSTM(units // 2))
    model.add(Dropout(dropout_rate))
    model.add(BatchNormalization())
    model.add(Dense(32, activation='relu'))
    model.add(Dense(horizon))
    model.compile(optimizer=Adam(learning_rate), loss='mse', metrics=['mae'])
    return model


def get_callbacks(name):
    return [
        EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=1),
        ModelCheckpoint(f'../models/{name}_best.h5', monitor='val_loss', save_best_only=True),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=7, min_lr=1e-6, verbose=1),
    ]

# Summary
build_simple_rnn(horizon=1).summary()
print()
build_lstm(horizon=1).summary()

---
## 7️⃣ Helpers — Metrics & Plots

In [ ]:
def mape(y_true, y_pred):
    y_true = np.where(y_true == 0, 1e-9, y_true)
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

def print_metrics(y_true, y_pred, label):
    if y_true.ndim > 1: y_true = y_true[:, 0]
    if y_pred.ndim > 1: y_pred = y_pred[:, 0]
    mse  = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae  = mean_absolute_error(y_true, y_pred)
    mpe  = mape(y_true, y_pred)
    print(f'  [{label}]')
    print(f'    MSE  = {mse:.6f}')
    print(f'    RMSE = {rmse:.6f}')
    print(f'    MAE  = {mae:.6f}')
    print(f'    MAPE = {mpe:.2f}%')
    return {'label': label, 'MSE': mse, 'RMSE': rmse, 'MAE': mae, 'MAPE%': mpe}

def inv_scale(arr):
    if arr.ndim == 1: arr = arr.reshape(-1, 1)
    return scaler.inverse_transform(arr[:, :1]).ravel()

def plot_pred(actual, rnn_pred, lstm_pred, horizon):
    n = min(300, len(actual))
    fig, ax = plt.subplots(figsize=(14, 5))
    ax.plot(actual[-n:],    label='Actual',    color='black',      linewidth=1.2)
    ax.plot(rnn_pred[-n:],  label='SimpleRNN', color='dodgerblue', linewidth=1.0, alpha=0.85)
    ax.plot(lstm_pred[-n:], label='LSTM',      color='tomato',     linewidth=1.0, alpha=0.85)
    ax.set_title(f'Actual vs Predicted (USD) — Horizon {horizon}d')
    ax.set_xlabel('Test Sample'); ax.set_ylabel('Price (USD)')
    ax.legend(); plt.tight_layout()
    plt.savefig(f'../reports/pred_h{horizon}.png', dpi=150)
    plt.show()

def plot_loss(rnn_h, lstm_h, horizon):
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    for ax, hist, name in zip(axes, [rnn_h, lstm_h], ['SimpleRNN', 'LSTM']):
        ax.plot(hist.history['loss'],     label='Train')
        ax.plot(hist.history['val_loss'], label='Val', linestyle='--')
        ax.set_title(f'{name} Loss — h={horizon}d')
        ax.set_xlabel('Epoch'); ax.set_ylabel('MSE'); ax.legend()
    plt.tight_layout()
    plt.savefig(f'../reports/loss_h{horizon}.png', dpi=150)
    plt.show()

print('✅ Helper functions defined')

---
## 8️⃣ Train & Evaluate — Horizon 1 Day

In [ ]:
H = 1
X, y         = create_sequences(scaled, WINDOW, H)
Xtr, Xte, ytr, yte = ts_split(X, y)
print(f'Train: {Xtr.shape}  Test: {Xte.shape}')

In [ ]:
# ── SimpleRNN ─────────────────────────────────────────────
rnn1 = build_simple_rnn(horizon=H)
rnn1_hist = rnn1.fit(
    Xtr, ytr, epochs=100, batch_size=32,
    validation_split=0.1,
    callbacks=get_callbacks('SimpleRNN_h1'),
    verbose=1
)
rnn1.save('../models/SimpleRNN_h1_final.h5')
print('✅ SimpleRNN h=1 saved')

In [ ]:
# ── LSTM ──────────────────────────────────────────────────
lstm1 = build_lstm(horizon=H)
lstm1_hist = lstm1.fit(
    Xtr, ytr, epochs=100, batch_size=32,
    validation_split=0.1,
    callbacks=get_callbacks('LSTM_h1'),
    verbose=1
)
lstm1.save('../models/LSTM_h1_final.h5')
print('✅ LSTM h=1 saved')

In [ ]:
plot_loss(rnn1_hist, lstm1_hist, H)

pred_rnn1  = rnn1.predict(Xte,  verbose=0)
pred_lstm1 = lstm1.predict(Xte, verbose=0)

print('\n── Metrics (Scaled Space) ───────────────')
m_rnn1  = print_metrics(yte, pred_rnn1,  'SimpleRNN h=1')
m_lstm1 = print_metrics(yte, pred_lstm1, 'LSTM      h=1')

plot_pred(inv_scale(yte), inv_scale(pred_rnn1), inv_scale(pred_lstm1), H)

---
## 9️⃣ Train & Evaluate — Horizon 5 Days

In [ ]:
H = 5
X5, y5 = create_sequences(scaled, WINDOW, H)
Xtr5, Xte5, ytr5, yte5 = ts_split(X5, y5)

rnn5 = build_simple_rnn(horizon=H)
rnn5_hist = rnn5.fit(Xtr5, ytr5, epochs=100, batch_size=32,
    validation_split=0.1, callbacks=get_callbacks('SimpleRNN_h5'), verbose=1)
rnn5.save('../models/SimpleRNN_h5_final.h5')

lstm5 = build_lstm(horizon=H)
lstm5_hist = lstm5.fit(Xtr5, ytr5, epochs=100, batch_size=32,
    validation_split=0.1, callbacks=get_callbacks('LSTM_h5'), verbose=1)
lstm5.save('../models/LSTM_h5_final.h5')

In [ ]:
plot_loss(rnn5_hist, lstm5_hist, H)
pred_rnn5  = rnn5.predict(Xte5,  verbose=0)
pred_lstm5 = lstm5.predict(Xte5, verbose=0)
print('\n── Metrics (Scaled Space) ───────────────')
m_rnn5  = print_metrics(yte5, pred_rnn5,  'SimpleRNN h=5')
m_lstm5 = print_metrics(yte5, pred_lstm5, 'LSTM      h=5')
plot_pred(inv_scale(yte5), inv_scale(pred_rnn5), inv_scale(pred_lstm5), H)

---
## 🔟 Train & Evaluate — Horizon 10 Days

In [ ]:
H = 10
X10, y10 = create_sequences(scaled, WINDOW, H)
Xtr10, Xte10, ytr10, yte10 = ts_split(X10, y10)

rnn10 = build_simple_rnn(horizon=H)
rnn10_hist = rnn10.fit(Xtr10, ytr10, epochs=100, batch_size=32,
    validation_split=0.1, callbacks=get_callbacks('SimpleRNN_h10'), verbose=1)
rnn10.save('../models/SimpleRNN_h10_final.h5')

lstm10 = build_lstm(horizon=H)
lstm10_hist = lstm10.fit(Xtr10, ytr10, epochs=100, batch_size=32,
    validation_split=0.1, callbacks=get_callbacks('LSTM_h10'), verbose=1)
lstm10.save('../models/LSTM_h10_final.h5')

In [ ]:
plot_loss(rnn10_hist, lstm10_hist, H)
pred_rnn10  = rnn10.predict(Xte10, verbose=0)
pred_lstm10 = lstm10.predict(Xte10, verbose=0)
print('\n── Metrics (Scaled Space) ───────────────')
m_rnn10  = print_metrics(yte10, pred_rnn10,  'SimpleRNN h=10')
m_lstm10 = print_metrics(yte10, pred_lstm10, 'LSTM      h=10')
plot_pred(inv_scale(yte10), inv_scale(pred_rnn10), inv_scale(pred_lstm10), H)

---
## 1️⃣1️⃣ Hyperparameter Tuning — GridSearchCV (LSTM, h=1)

In [ ]:
"""
GridSearchCV tries all combinations of:
  units         : [32, 64]     → number of LSTM neurons
  dropout_rate  : [0.1, 0.2]  → overfitting prevention
  learning_rate : [0.001, 0.0005] → Adam optimizer step size

Total combinations: 2 × 2 × 2 = 8 configs × 3 folds = 24 runs
"""

def make_lstm_gs(units=64, dropout_rate=0.2, learning_rate=0.001):
    return build_lstm(units=units, dropout_rate=dropout_rate,
                      learning_rate=learning_rate, horizon=1)

reg = KerasRegressor(model=make_lstm_gs, epochs=30, batch_size=32, verbose=0)

param_grid = {
    'model__units':         [32, 64],
    'model__dropout_rate':  [0.1, 0.2],
    'model__learning_rate': [0.001, 0.0005],
}

gs = GridSearchCV(reg, param_grid, cv=3,
                  scoring='neg_mean_squared_error', n_jobs=1, verbose=2)
gs.fit(Xtr1, ytr1.ravel())

print('\n✅ Best Params:', gs.best_params_)
print('   Best CV MSE:', round(-gs.best_score_, 6))

In [ ]:
# ── Show GridSearch results table ─────────────────────────
gs_df = pd.DataFrame(gs.cv_results_).sort_values('rank_test_score')
gs_df[['param_model__units','param_model__dropout_rate',
       'param_model__learning_rate','mean_test_score','rank_test_score']].head(8)

In [ ]:
# ── Train final tuned LSTM ────────────────────────────────
bp = gs.best_params_
best_lstm = build_lstm(
    units=bp['model__units'],
    dropout_rate=bp['model__dropout_rate'],
    learning_rate=bp['model__learning_rate'],
    horizon=1
)
best_lstm.fit(
    Xtr1, ytr1, epochs=100, batch_size=32,
    validation_split=0.1,
    callbacks=get_callbacks('LSTM_tuned_h1'),
    verbose=1
)
best_lstm.save('../models/LSTM_tuned_h1_final.h5')

pred_best = best_lstm.predict(Xte1, verbose=0)
print('\n── Tuned LSTM Metrics ───────────────────')
print_metrics(yte1, pred_best, 'Tuned LSTM h=1')

---
## 1️⃣2️⃣ Model Comparison Summary

In [ ]:
all_metrics = [
    {**m_rnn1,   'Horizon': 1,  'Model': 'SimpleRNN'},
    {**m_lstm1,  'Horizon': 1,  'Model': 'LSTM'},
    {**m_rnn5,   'Horizon': 5,  'Model': 'SimpleRNN'},
    {**m_lstm5,  'Horizon': 5,  'Model': 'LSTM'},
    {**m_rnn10,  'Horizon': 10, 'Model': 'SimpleRNN'},
    {**m_lstm10, 'Horizon': 10, 'Model': 'LSTM'},
]

summary = pd.DataFrame(all_metrics)[['Model','Horizon','MSE','RMSE','MAE','MAPE%']]
summary = summary.sort_values(['Horizon','Model'])
summary.to_csv('../reports/all_metrics.csv', index=False)
print('✅ Summary saved')
summary

In [ ]:
# ── RMSE Bar Chart ────────────────────────────────────────
pivot = summary.pivot(index='Horizon', columns='Model', values='RMSE')
ax = pivot.plot(kind='bar', figsize=(10, 5), color=['dodgerblue','tomato'])
ax.set_title('RMSE Comparison: SimpleRNN vs LSTM', fontsize=13)
ax.set_xlabel('Forecast Horizon (days)'); ax.set_ylabel('RMSE (scaled)')
ax.legend(title='Model'); plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('../reports/comparison.png', dpi=150)
plt.show()

---
## 1️⃣3️⃣ Future Price Forecast (Rolling Prediction)

In [ ]:
N_DAYS   = 30
model_fc = load_model('../models/LSTM_h1_final.h5', compile=False)

seed    = scaled[-WINDOW:].reshape(1, WINDOW, 1)
preds   = []
current = seed.copy()

for _ in range(N_DAYS):
    p = model_fc.predict(current, verbose=0)[0, 0]
    preds.append(p)
    current = np.concatenate([current[:, 1:, :], [[[p]]]], axis=1)

preds_usd    = scaler.inverse_transform(np.array(preds).reshape(-1,1)).ravel()
last_date    = df.index[-1]
future_dates = pd.bdate_range(start=last_date, periods=N_DAYS + 1)[1:]

fc_df = pd.DataFrame({'Date': future_dates, 'Forecast ($)': preds_usd.round(2)})
print(fc_df.to_string(index=False))

In [ ]:
recent = df['Adj Close'].tail(60)
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(recent.index, recent.values, label='Historical (last 60d)', color='black', linewidth=1.2)
ax.plot(future_dates, preds_usd,     label='LSTM 30-Day Forecast',
        color='dodgerblue', linewidth=1.5, marker='o', markersize=4)
ax.axvline(last_date, color='red', linestyle='--', alpha=0.5, label='Forecast Start')
ax.set_title('Tesla — 30-Day Future Price Forecast (LSTM)')
ax.set_ylabel('Price (USD)'); ax.legend()
plt.tight_layout()
plt.savefig('../reports/future_forecast.png', dpi=150)
plt.show()

---
## 1️⃣4️⃣ Conclusion & Insights

### Results Summary
| Finding | Detail |
|---|---|
| **Best Model** | LSTM consistently outperforms SimpleRNN |
| **Best Horizon** | 1-day forecast has lowest error |
| **Why LSTM wins** | Gated memory solves vanishing gradient problem |
| **Scaling** | MinMaxScaler essential for stable RNN training |
| **Missing values** | Forward-fill is correct for time-series data |

### Limitations
- Only uses price history — ignores news, earnings, macroeconomics
- Stock markets have random events no model can fully predict
- Past performance ≠ future results

### Future Improvements
1. Add Twitter/news **sentiment analysis** as input features
2. Try **GRU** or **Transformer** models
3. Use **multi-variate input** (Volume + technical indicators)
4. Implement **walk-forward validation**
5. Compare with **ARIMA** baseline

---
*Tesla Stock Price Prediction | SimpleRNN & LSTM | Deep Learning Project*